# Day 4 — PySpark: DataFrame Aggregations
**Topics:** groupBy, agg, sum, count, avg, min, max, HAVING equivalent via filter

In [2]:
import os
os.environ['JAVA_HOME']             = 'C:/Program Files/DBeaver/jre'
os.environ['PYSPARK_PYTHON']        = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Day4_Aggregations') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

# Sample data — same as SQL notebook
sales_data = [
    ('S001','P001','North','2024-01-05', 10, 29.99),
    ('S002','P002','South','2024-01-07',  5, 49.99),
    ('S003','P001','East', '2024-01-10', 20, 29.99),
    ('S004','P003','West', '2024-01-12', 15, 99.99),
    ('S005','P002','North','2024-01-15',  8, 49.99),
    ('S006','P001','South','2024-02-01', 30, 29.99),
    ('S007','P003','East', '2024-02-03',  2, 99.99),
    ('S008','P004','West', '2024-02-08', 25, 14.99),
    ('S009','P004','North','2024-02-11', 40, 14.99),
    ('S010','P002','East', '2024-02-14', 12, 49.99),
    ('S011','P001','West', '2024-03-01', 18, 29.99),
    ('S012','P003','South','2024-03-05',  7, 99.99),
    ('S013','P004','East', '2024-03-09', 22, 14.99),
    ('S014','P002','West', '2024-03-12',  3, 49.99),
    ('S015','P001','North','2024-03-20', 14, 29.99),
]

schema = StructType([
    StructField('sale_id',    StringType(),  False),
    StructField('product_id', StringType(),  True),
    StructField('region',     StringType(),  True),
    StructField('sale_date',  StringType(),  True),
    StructField('quantity',   IntegerType(), True),
    StructField('unit_price', DoubleType(),  True),
])

df_sales = spark.createDataFrame(sales_data, schema=schema)
df_sales.printSchema()
df_sales.show(5)
print('Total rows:', df_sales.count())

Spark version: 3.5.6
root
 |-- sale_id: string (nullable = false)
 |-- product_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)

+-------+----------+------+----------+--------+----------+
|sale_id|product_id|region| sale_date|quantity|unit_price|
+-------+----------+------+----------+--------+----------+
|   S001|      P001| North|2024-01-05|      10|     29.99|
|   S002|      P002| South|2024-01-07|       5|     49.99|
|   S003|      P001|  East|2024-01-10|      20|     29.99|
|   S004|      P003|  West|2024-01-12|      15|     99.99|
|   S005|      P002| North|2024-01-15|       8|     49.99|
+-------+----------+------+----------+--------+----------+
only showing top 5 rows

Total rows: 15


## 1. count — total rows

In [3]:
# count('*') — all rows; count('col') — non-NULL only
df_sales.agg(
    F.count('*').alias('total_rows'),
    F.count('region').alias('non_null_region'),
    F.countDistinct('region').alias('distinct_regions'),
).show()

+----------+---------------+----------------+
|total_rows|non_null_region|distinct_regions|
+----------+---------------+----------------+
|        15|             15|               4|
+----------+---------------+----------------+



## 2. sum — total quantity and revenue

In [4]:
# Global aggregation — no groupBy
df_sales.agg(
    F.sum('quantity').alias('total_units'),
    F.round(
        F.sum(F.col('quantity') * F.col('unit_price')), 2
    ).alias('total_revenue'),
).show()

+-----------+-------------+
|total_units|total_revenue|
+-----------+-------------+
|        231|      7862.69|
+-----------+-------------+



## 3. groupBy + agg — revenue per region

In [ ]:
# Option A: withColumn first (more readable)
df_revenue = (
    df_sales
    .withColumn('revenue', F.col('quantity') * F.col('unit_price'))
    .groupBy('region')
    .agg(
        F.round(F.sum('revenue'), 2).alias('total_revenue'),
        F.sum('quantity').alias('total_units'),
        F.count('*').alias('num_sales'),
    )
    .orderBy('total_revenue', ascending=False)
)
df_revenue.show()

## 4. min / max / avg per group

In [ ]:
df_prices = (
    df_sales
    .groupBy('region')
    .agg(
        F.min('unit_price').alias('min_price'),
        F.max('unit_price').alias('max_price'),
        F.round(F.avg('unit_price'), 2).alias('avg_price'),
    )
    .orderBy('region')
)
df_prices.show()

## 5. HAVING equivalent — filter() after agg()

In [ ]:
# SQL HAVING SUM(quantity) > 50
# PySpark: groupBy → agg → filter
df_top = (
    df_sales
    .groupBy('product_id')
    .agg(F.sum('quantity').alias('total_units_sold'))
    .filter(F.col('total_units_sold') > 50)     # ← this IS the HAVING
    .orderBy('total_units_sold', ascending=False)
)
print('Products with >50 units sold:')
df_top.show()

## 6. All five aggregates in one agg() call

In [ ]:
df_summary = (
    df_sales
    .groupBy('region')
    .agg(
        F.count('*').alias('num_transactions'),
        F.min('unit_price').alias('min_price'),
        F.max('unit_price').alias('max_price'),
        F.round(F.avg('unit_price'), 2).alias('avg_price'),
        F.sum('quantity').alias('total_qty'),
    )
    .orderBy('region')
)
df_summary.show()

## 7. groupBy multiple columns

In [ ]:
# Revenue per region + product
df_multi = (
    df_sales
    .withColumn('revenue', F.col('quantity') * F.col('unit_price'))
    .groupBy('region', 'product_id')
    .agg(
        F.round(F.sum('revenue'), 2).alias('revenue'),
        F.sum('quantity').alias('units'),
    )
    .orderBy('region', F.col('revenue').desc())
)
df_multi.show()

In [ ]:
spark.stop()
print('Spark stopped.')